# 02 · Dimensionality Reduction & Clustering
### Lifestyle Archetypes and Problematic Internet Use — Pipeline Notebook 2 of 5

**This notebook covers:**
6. Dimensionality reduction (UMAP)
7. Clustering comparison (K-means, Agglomerative, DBSCAN)

**Loads:** `01_data_preparation.pkl` (from `01_data_preparation.ipynb`)
**Produces:** `02_dimensionality_reduction_clustering.pkl`, consumed by `03_validation_stability.ipynb` and `04_archetype_characterisation.ipynb`.

**Responsible-analysis boundary (carried through every notebook in this pipeline):** this is an unsupervised, exploratory segmentation task. Clusters describe population-level lifestyle patterns, not individual diagnoses.

## Setup & Environment

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import dendrogram, linkage

import umap.umap_ as umap


In [ ]:
!pip -q install umap-learn scikit-learn-extra


### Load artifacts from notebook 01

In [ ]:
ARTIFACT_DIR = Path("/kaggle/working/artifacts")

with open(ARTIFACT_DIR / "01_data_preparation.pkl", "rb") as f:
    artifact_01 = pickle.load(f)

model_df = artifact_01["model_df"]
X_imputed = artifact_01["X_imputed"]
CLUSTER_FEATURES = artifact_01["CLUSTER_FEATURES"]
CHARACTERISATION_VARS = artifact_01["CHARACTERISATION_VARS"]

print(f"Loaded model_df {model_df.shape}, X_imputed {X_imputed.shape}")


## 6. Dimensionality Reduction (UMAP)

Features are standardised first (z-scored), since they are on very different scales (hours, g, kg/m^2, questionnaire totals). We then explore how UMAP's `n_neighbors` (local vs global structure) and `min_dist` (how tightly points are allowed to pack) affect the shape of the embedding before committing to final parameters.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)
X_scaled = pd.DataFrame(X_scaled, columns=CLUSTER_FEATURES, index=X_imputed.index)

# quick PCA sanity check - how much variance do the raw features already carry
pca = PCA(random_state=42).fit(X_scaled)
plt.figure(figsize=(6, 4))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker="o")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA on standardised lifestyle features")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
neighbor_grid = [5, 15, 50]
min_dist_grid = [0.0, 0.5]

fig, axes = plt.subplots(len(neighbor_grid), len(min_dist_grid), figsize=(10, 14))
for i, n_neighbors in enumerate(neighbor_grid):
    for j, min_dist in enumerate(min_dist_grid):
        reducer = umap.UMAP(
            n_neighbors=n_neighbors,
            min_dist=min_dist,
            n_components=2,
            random_state=42,
        )
        emb = reducer.fit_transform(X_scaled)
        ax = axes[i, j]
        ax.scatter(emb[:, 0], emb[:, 1], s=6, alpha=0.5, color="#4C72B0")
        ax.set_title(f"n_neighbors={n_neighbors}, min_dist={min_dist}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


**Parameter choice rationale:** a moderate `n_neighbors` (15) balances local detail against the global structure needed for population-level archetypes (very small `n_neighbors` fragments the embedding into noisy local clusters; very large `n_neighbors` over-smooths genuine subgroup structure). A small but non-zero `min_dist` (0.1) keeps groups visually separable without artificially collapsing genuinely continuous variation into artefactual tight blobs. We fit both a 2D embedding (for visualisation) and a higher-dimensional embedding (for clustering itself, which benefits from retaining more structure than 2D allows).

In [ ]:
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1

reducer_2d = umap.UMAP(
    n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST, n_components=2, random_state=42
)
embedding_2d = reducer_2d.fit_transform(X_scaled)

reducer_5d = umap.UMAP(
    n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST, n_components=5, random_state=42
)
embedding_5d = reducer_5d.fit_transform(X_scaled)

print("2D embedding shape:", embedding_2d.shape)
print("5D embedding shape (used for clustering):", embedding_5d.shape)


## 7. Clustering Comparison

We compare three families of clustering algorithm on the 5D UMAP embedding, each with a different bias: K-means (spherical, evenly-sized clusters), Agglomerative/hierarchical (nested, variable-shape clusters), and DBSCAN (density-based, can flag noise/outliers rather than forcing every child into a cluster). Testing more than one method guards against reporting an artefact of a single algorithm's assumptions.

In [ ]:
# --- K-means: elbow + silhouette across k ---
k_range = range(2, 9)
inertias, sil_scores_km = [], []
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(embedding_5d)
    inertias.append(km.inertia_)
    sil_scores_km.append(silhouette_score(embedding_5d, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), inertias, marker="o")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia"); axes[0].set_title("K-means elbow")
axes[1].plot(list(k_range), sil_scores_km, marker="o", color="darkorange")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette score"); axes[1].set_title("K-means silhouette vs k")
plt.tight_layout()
plt.show()

best_k = list(k_range)[int(np.argmax(sil_scores_km))]
print(f"Silhouette-selected k for K-means: {best_k}")


In [ ]:
kmeans_final = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit(embedding_5d)
labels_kmeans = kmeans_final.labels_


In [ ]:
# --- Agglomerative (Ward linkage) ---
Z = linkage(embedding_5d, method="ward")
plt.figure(figsize=(12, 4))
dendrogram(Z, truncate_mode="lastp", p=30, show_leaf_counts=True)
plt.title("Agglomerative clustering (Ward linkage) - truncated dendrogram")
plt.xlabel("Cluster size"); plt.ylabel("Distance")
plt.show()

sil_scores_agg = []
for k in k_range:
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward").fit(embedding_5d)
    sil_scores_agg.append(silhouette_score(embedding_5d, agg.labels_))

plt.figure(figsize=(6, 4))
plt.plot(list(k_range), sil_scores_agg, marker="o", color="seagreen")
plt.xlabel("k"); plt.ylabel("Silhouette score"); plt.title("Agglomerative silhouette vs k")
plt.show()

best_k_agg = list(k_range)[int(np.argmax(sil_scores_agg))]
print(f"Silhouette-selected k for Agglomerative: {best_k_agg}")
agg_final = AgglomerativeClustering(n_clusters=best_k_agg, linkage="ward").fit(embedding_5d)
labels_agg = agg_final.labels_


In [ ]:
# --- DBSCAN: eps tuning via k-distance graph ---
MIN_SAMPLES = 10
neighbors = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(embedding_5d)
distances, _ = neighbors.kneighbors(embedding_5d)
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(6, 4))
plt.plot(k_distances)
plt.xlabel("Points sorted by distance"); plt.ylabel(f"{MIN_SAMPLES}-NN distance")
plt.title("DBSCAN eps selection (k-distance graph - look for the 'elbow')")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# Elbow read from the plot above - adjust EPS if the elbow sits elsewhere for your run
EPS = float(np.percentile(k_distances, 90))
print(f"Selected eps (90th percentile of k-distance, as an elbow proxy): {EPS:.3f}")

dbscan_final = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES).fit(embedding_5d)
labels_dbscan = dbscan_final.labels_
n_clusters_dbscan = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
n_noise = int((labels_dbscan == -1).sum())
print(f"DBSCAN found {n_clusters_dbscan} clusters and flagged {n_noise} points as noise ({n_noise/len(labels_dbscan):.1%})")


## Save artifacts for downstream notebooks

In [ ]:
artifact = {
    "X_scaled": X_scaled,
    "embedding_2d": embedding_2d,
    "embedding_5d": embedding_5d,
    "k_range": list(k_range),
    "k_distances": k_distances,
    "UMAP_N_NEIGHBORS": UMAP_N_NEIGHBORS,
    "UMAP_MIN_DIST": UMAP_MIN_DIST,
    "best_k": best_k,
    "best_k_agg": best_k_agg,
    "EPS": EPS,
    "MIN_SAMPLES": MIN_SAMPLES,
    "labels_kmeans": labels_kmeans,
    "labels_agg": labels_agg,
    "labels_dbscan": labels_dbscan,
}

with open(ARTIFACT_DIR / "02_dimensionality_reduction_clustering.pkl", "wb") as f:
    pickle.dump(artifact, f)

print(f"Saved artifact -> {ARTIFACT_DIR / '02_dimensionality_reduction_clustering.pkl'}")
